# 04 Scaling Laws

Builds a scaling-run inventory, estimates log-log relationships where enough non-collinear data exists, and explicitly refuses to draw scaling-law conclusions from insufficient or smoke-only grids.

In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'wwgpt').exists():
            return p
    raise RuntimeError('Could not locate repository root')

REPO_ROOT = find_repo_root(Path.cwd())
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wwgpt.analysis import (
    completed_runs, discover_experiment_runs, normalize_metrics, terminal_results,
    add_generalization_measures, vocab_size_from_artifacts, summary, align_curves,
    paired_curve_differences,
)

RESULTS_ROOT = Path(globals().get('RESULTS_ROOT', REPO_ROOT / 'results'))
ANALYSIS_DIR = Path(globals().get('ANALYSIS_DIR', RESULTS_ROOT / 'notebook_analysis'))
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository: {REPO_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Notebook outputs: {ANALYSIS_DIR}')


## Scaling run inventory

In [ ]:
completed=completed_runs(RESULTS_ROOT, scientific_only=False)
rows=[]
for run in completed:
    man=json.loads((run/'manifest.json').read_text())
    m=normalize_metrics(pd.read_csv(run/'metrics.csv')) if (run/'metrics.csv').exists() else pd.DataFrame()
    last=m.sort_values('step').tail(1) if 'step' in m and not m.empty else m.tail(1)
    report=man.get('parameter_report', {})
    rows.append({'run_dir': str(run), 'optimizer': man.get('optimizer'), 'seed': man.get('seed'), 'pair_id': man.get('pair_id'), 'level': man.get('level'), 'token_multiplier': man.get('token_multiplier'), 'target_tokens': man.get('target_tokens'), 'total_parameters': report.get('total_parameters') or man.get('total_parameters'), 'non_embedding_parameters': report.get('non_embedding_parameters'), 'valid_for_science': man.get('valid_for_science'), 'smoke_test': man.get('smoke_test'), 'final_val_loss': float(last['validation_loss'].iloc[0]) if not last.empty and 'validation_loss' in last else np.nan})
scaling=pd.DataFrame(rows)
scaling.to_csv(ANALYSIS_DIR/'scaling_run_inventory.csv', index=False)
if scaling.empty:
    display(Markdown('No completed runs found under RESULTS_ROOT.'))
else:
    display(scaling)
    display(scaling.groupby(['level','token_multiplier','optimizer','valid_for_science','smoke_test'], dropna=False).size().reset_index(name='runs'))


## Fit simple empirical surfaces

In [ ]:
fit_rows=[]
fit_data=scaling[(scaling['valid_for_science']==True) & (scaling['smoke_test']!=True)].dropna(subset=['final_val_loss','total_parameters','target_tokens']) if not scaling.empty else pd.DataFrame()
for opt,g in fit_data.groupby('optimizer') if not fit_data.empty else []:
    d=g[(g['final_val_loss']>0) & (g['total_parameters']>0) & (g['target_tokens']>0)].copy()
    rank=np.linalg.matrix_rank(np.column_stack([np.ones(len(d)), np.log(d['total_parameters']), np.log(d['target_tokens'])])) if len(d) else 0
    if len(d) >= 4 and rank >= 3:
        X=np.column_stack([np.ones(len(d)), np.log(d['total_parameters']), np.log(d['target_tokens'])])
        y=np.log(d['final_val_loss'])
        beta=np.linalg.lstsq(X, y, rcond=None)[0]
        pred=X@beta
        fit_rows.append({'optimizer': opt, 'runs': len(d), 'status': 'fit', 'intercept': beta[0], 'parameter_exponent': beta[1], 'token_exponent': beta[2], 'rmse_log_loss': float(np.sqrt(np.mean((y-pred)**2)))})
    else:
        fit_rows.append({'optimizer': opt, 'runs': len(d), 'status': 'not_fit_insufficient_non_collinear_grid'})
fits=pd.DataFrame(fit_rows) if fit_rows else pd.DataFrame([{'status':'not_fit_no_scientific_scaling_runs'}])
fits.to_csv(ANALYSIS_DIR/'scaling_fit_results.csv', index=False)
display(fits)


## Visual diagnostics

In [ ]:
if not scaling.empty:
    plot_data=scaling.dropna(subset=['total_parameters','final_val_loss'])
    if not plot_data.empty:
        plt.figure(figsize=(8,5))
        for (opt, valid), g in plot_data.groupby(['optimizer','valid_for_science'], dropna=False):
            plt.scatter(g['total_parameters'], g['final_val_loss'], label=f'{opt}, scientific={valid}', alpha=.8)
        plt.xscale('log'); plt.xlabel('total parameters'); plt.ylabel('final validation loss'); plt.legend(); plt.tight_layout()
        plt.savefig(ANALYSIS_DIR/'scaling_loss_vs_parameters.png', dpi=160); plt.show()
